**Классификация отношений между элементами аргументативной структуры**

*1. Установим необходимые библиотеки*

In [ ]:
!pip install -q -U sentence-transformers transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.3 MB/s eta 0:00:00


In [ ]:
import ast
import gc
import re
import numpy as np
import pandas as pd
import random
import torch

from datasets import load_dataset, Dataset, concatenate_datasets
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


*2. Загрузим датасет*

In [ ]:
ds_en = load_dataset("HiTZ/casimedicos-arg", "en")
ds_es = load_dataset("HiTZ/casimedicos-arg", "es")
ds_fr = load_dataset("HiTZ/casimedicos-arg", "fr")
ds_it = load_dataset("HiTZ/casimedicos-arg", "it")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_en_all.csv: 0.00B [00:00, ?B/s]

validation_en_all.csv: 0.00B [00:00, ?B/s]

test_en_all.csv: 0.00B [00:00, ?B/s]

...

validation_it_ordered.csv: 0.00B [00:00, ?B/s]

test_it_ordered.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/434 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/63 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/125 [00:00<?, ? examples/s]

In [ ]:
print("EN:", ds_en["train"].column_names)
print("ES:", ds_es["train"].column_names)
print("FR:", ds_fr["train"].column_names)
print("IT:", ds_it["train"].column_names)

EN: ['id', 'text', 'labels', 'relations']
ES: ['id', 'text', 'labels']
FR: ['id', 'text', 'labels']
IT: ['id', 'text', 'labels']


Здесь мы сталкиваемся с проблемой - отношения между компонентами аргументационной структуры указаны только в сабсете на английском языке.

*3. Поработаем с датасетом*

In [ ]:
def parse_py(s):
    return ast.literal_eval(s) if isinstance(s, str) else s

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())

def extract_components_from_bio(text_str: str, labels_str: str):
    """
    Возвращает список компонент (строки) в порядке появления.
    """
    tokens_segs = parse_py(text_str)     # list[list[str]]
    labels_segs = parse_py(labels_str)   # list[list[str]]
    if not isinstance(tokens_segs, list) or not isinstance(labels_segs, list):
        return []

    comps = []
    cur = []

    def flush():
        nonlocal cur
        if cur:
            comps.append(normalize_ws(" ".join(cur)))
            cur = []

    for toks, labs in zip(tokens_segs, labels_segs):
        if not isinstance(toks, list) or not isinstance(labs, list):
            flush()
            continue
        m = min(len(toks), len(labs))
        for tok, lab in zip(toks[:m], labs[:m]):
            tok = str(tok)
            if lab == "O":
                flush()
                continue
            if isinstance(lab, str) and lab.startswith("B-"):
                flush()
                cur = [tok]
            elif isinstance(lab, str) and lab.startswith("I-"):
                if cur:
                    cur.append(tok)
                else:
                    cur = [tok]
            else:
                flush()
        flush()

    return comps

In [ ]:
embed_model = SentenceTransformer("intfloat/multilingual-e5-base")

def e5_encode(texts):
    q = [f"query: {t}" for t in texts]
    return embed_model.encode(q, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

def e5_encode_passage(texts):
    p = [f"passage: {t}" for t in texts]
    return embed_model.encode(p, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

def best_match_index(query_text, passages, passages_emb):
    q_emb = e5_encode([query_text])[0]                 # (d,)
    sims = passages_emb @ q_emb                        # cosine because normalized
    idx = int(np.argmax(sims))
    return idx, float(sims[idx])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
def build_relations_for_doc(en_ex, tgt_ex, lang_name, sim_threshold=0.60):
    """
    Возвращает список relations для target языка в формате [[a,b,label], ...]
    """
    # EN components
    en_comps = extract_components_from_bio(en_ex["text"], en_ex["labels"])
    if len(en_comps) == 0:
        return []

    # Target components
    tgt_comps = extract_components_from_bio(tgt_ex["text"], tgt_ex["labels"])
    if len(tgt_comps) == 0:
        return []

    # embeddings for EN components once
    en_comp_emb = e5_encode_passage(en_comps)

    rels = parse_py(en_ex["relations"])  # list of [spanA, spanB, label]
    out = []

    counts_match = (len(en_comps) == len(tgt_comps))
    if not counts_match:
        tgt_comp_emb = e5_encode_passage(tgt_comps)

    for spanA, spanB, lab in rels:
        spanA = normalize_ws(str(spanA))
        spanB = normalize_ws(str(spanB))
        lab = str(lab).strip()

        i, sim_i = best_match_index(spanA, en_comps, en_comp_emb)
        j, sim_j = best_match_index(spanB, en_comps, en_comp_emb)

        # если очень слабое совпадение — пропускаем, чтобы не засорять silver labels
        if sim_i < sim_threshold or sim_j < sim_threshold:
            continue

        if counts_match and i < len(tgt_comps) and j < len(tgt_comps):
            a_t = tgt_comps[i]
            b_t = tgt_comps[j]
        else:
            # fallback: находим соответствующие компоненты прямо в target по смыслу
            a_idx, a_sim = best_match_index(en_comps[i], tgt_comps, tgt_comp_emb)
            b_idx, b_sim = best_match_index(en_comps[j], tgt_comps, tgt_comp_emb)
            if a_sim < sim_threshold or b_sim < sim_threshold:
                continue
            a_t = tgt_comps[a_idx]
            b_t = tgt_comps[b_idx]

        out.append([a_t, b_t, lab])

    return out

In [ ]:
def add_relations_column(ds_en_split, ds_tgt_split, lang_name, sim_threshold=0.60):
    tgt_by_id = {ex["id"]: ex for ex in ds_tgt_split}

    new_rows = {"id": [], "text": [], "labels": [], "relations_gen": []}
    kept = 0
    total = 0

    for en_ex in ds_en_split:
        doc_id = en_ex["id"]
        if doc_id not in tgt_by_id:
            continue
        tgt_ex = tgt_by_id[doc_id]

        rels = build_relations_for_doc(en_ex, tgt_ex, lang_name, sim_threshold=sim_threshold)
        total += 1
        if len(rels) == 0:
            rels_str = "[]"
        else:
            kept += 1
            rels_str = str(rels)

        new_rows["id"].append(tgt_ex["id"])
        new_rows["text"].append(tgt_ex["text"])
        new_rows["labels"].append(tgt_ex["labels"])
        new_rows["relations_gen"].append(rels_str)

    print(f"[{lang_name}] docs processed: {total}, docs with >=1 relation: {kept}")
    return Dataset.from_dict(new_rows)

In [ ]:
ds_en = load_dataset("HiTZ/casimedicos-arg", "en")
ds_es = load_dataset("HiTZ/casimedicos-arg", "es")
ds_fr = load_dataset("HiTZ/casimedicos-arg", "fr")
ds_it = load_dataset("HiTZ/casimedicos-arg", "it")

es_train = add_relations_column(ds_en["train"], ds_es["train"], "es", sim_threshold=0.60)
es_val   = add_relations_column(ds_en["validation"], ds_es["validation"], "es", sim_threshold=0.60)
es_test  = add_relations_column(ds_en["test"], ds_es["test"], "es", sim_threshold=0.60)

fr_train = add_relations_column(ds_en["train"], ds_fr["train"], "fr", sim_threshold=0.60)
fr_val   = add_relations_column(ds_en["validation"], ds_fr["validation"], "fr", sim_threshold=0.60)
fr_test  = add_relations_column(ds_en["test"], ds_fr["test"], "fr", sim_threshold=0.60)

it_train = add_relations_column(ds_en["train"], ds_it["train"], "it", sim_threshold=0.60)
it_val   = add_relations_column(ds_en["validation"], ds_it["validation"], "it", sim_threshold=0.60)
it_test  = add_relations_column(ds_en["test"], ds_it["test"], "it", sim_threshold=0.60)

print(es_train[0]["relations_gen"])

[es] docs processed: 434, docs with >=1 relation: 379
[es] docs processed: 63, docs with >=1 relation: 55
[es] docs processed: 125, docs with >=1 relation: 116
[fr] docs processed: 434, docs with >=1 relation: 379
[fr] docs processed: 63, docs with >=1 relation: 55
[fr] docs processed: 125, docs with >=1 relation: 116
[it] docs processed: 434, docs with >=1 relation: 379
[it] docs processed: 63, docs with >=1 relation: 55
[it] docs processed: 125, docs with >=1 relation: 116
[['Es indudable que la relación médico-paciente hoy día trasciende el entorno físico de la consulta.', 'Contestarle que es importante mantener unos ciertos limites profesionales entre pacientes y facultativos y que, desafortunadamente, si hace la petición no podrá aceptarla, por lo que mejor que no lo haga.', 'Support'], ['Pero no debemos olvidar que debemos mantener en ella los mismos códigos éticos y profesionales que en el entorno real.', 'Contestarle que es importante mantener unos ciertos limites profesionales

In [ ]:
print(es_train[0]["relations_gen"])

[['Es indudable que la relación médico-paciente hoy día trasciende el entorno físico de la consulta.', 'Contestarle que es importante mantener unos ciertos limites profesionales entre pacientes y facultativos y que, desafortunadamente, si hace la petición no podrá aceptarla, por lo que mejor que no lo haga.', 'Support'], ['Pero no debemos olvidar que debemos mantener en ella los mismos códigos éticos y profesionales que en el entorno real.', 'Contestarle que es importante mantener unos ciertos limites profesionales entre pacientes y facultativos y que, desafortunadamente, si hace la petición no podrá aceptarla, por lo que mejor que no lo haga.', 'Support'], ['El ejercicio clínico de la medicina mediante consultas exclusivamente por carta, teléfono, radio, prensa o internet, es contrario a las normas deontológicas.', 'Contestarle que es importante mantener unos ciertos limites profesionales entre pacientes y facultativos y que, desafortunadamente, si hace la petición no podrá aceptarla,

In [ ]:
def flatten_split(ds, rel_column, lang_name):
    rows = {"text_a": [], "text_b": [], "label": [], "lang": [], "doc_id": []}
    for ex in ds:
        rels = ast.literal_eval(ex[rel_column])
        for a, b, lab in rels:
            rows["text_a"].append(a.strip())
            rows["text_b"].append(b.strip())
            rows["label"].append(str(lab).strip())
            rows["lang"].append(lang_name)
            rows["doc_id"].append(ex["id"])
    return Dataset.from_dict(rows)

In [ ]:
# EN
train_en = flatten_split(ds_en["train"], "relations", "en")
val_en   = flatten_split(ds_en["validation"], "relations", "en")
test_en  = flatten_split(ds_en["test"], "relations", "en")

# ES/FR/IT (используем relations_gen)
train_es = flatten_split(es_train, "relations_gen", "es")
val_es   = flatten_split(es_val, "relations_gen", "es")
test_es  = flatten_split(es_test, "relations_gen", "es")

train_fr = flatten_split(fr_train, "relations_gen", "fr")
val_fr   = flatten_split(fr_val, "relations_gen", "fr")
test_fr  = flatten_split(fr_test, "relations_gen", "fr")

train_it = flatten_split(it_train, "relations_gen", "it")
val_it   = flatten_split(it_val, "relations_gen", "it")
test_it  = flatten_split(it_test, "relations_gen", "it")

# объединяем
train_flat = concatenate_datasets([train_en, train_es, train_fr, train_it])
val_flat   = concatenate_datasets([val_en, val_es, val_fr, val_it])
test_flat  = concatenate_datasets([test_en, test_es, test_fr, test_it])

print("Train size:", len(train_flat))
print("Val size:", len(val_flat))
print("Test size:", len(test_flat))
print("Labels:", sorted(set(train_flat["label"])))

Train size: 9536
Val size: 1484
Test size: 3016
Labels: ['Attack', 'Support']


Проверим, верно ли модель перенесла элементы аргументативной структуры и отношения между ними в другие сабсеты.

In [ ]:
def show_doc_relations(doc_id, max_rel=6):
    en_ex = next(x for x in ds_en["train"] if x["id"] == doc_id)
    fr_ex = next(x for x in fr_train if x["id"] == doc_id)

    en_rels = ast.literal_eval(en_ex["relations"])
    fr_rels = ast.literal_eval(fr_ex["relations_gen"])

    print("="*120)
    print("DOC:", doc_id)
    print(f"EN relations: {len(en_rels)} | FR relations_gen: {len(fr_rels)}")
    print("-"*120)

    k = min(max_rel, len(en_rels), len(fr_rels))
    for i in range(k):
        a_en, b_en, lab_en = en_rels[i]
        a_fr, b_fr, lab_fr = fr_rels[i]

        print(f"[{i}] LABEL EN={lab_en} | FR={lab_fr}")
        print("EN A:", a_en.strip())
        print("FR A:", a_fr.strip())
        print("EN B:", b_en.strip())
        print("FR B:", b_fr.strip())
        print("-"*120)

In [ ]:
from sentence_transformers import util

def show_doc_relations_matched(doc_id, max_rel=6):
    en_ex = next(x for x in ds_en["train"] if x["id"] == doc_id)
    fr_ex = next(x for x in fr_train if x["id"] == doc_id)

    en_rels = ast.literal_eval(en_ex["relations"])
    fr_rels = ast.literal_eval(fr_ex["relations_gen"])

    print("="*120)
    print("DOC:", doc_id)
    print(f"EN relations: {len(en_rels)} | FR relations_gen: {len(fr_rels)}")

    en_texts = [f"{a.strip()} || {b.strip()}" for a,b,_ in en_rels]
    fr_texts = [f"{a.strip()} || {b.strip()}" for a,b,_ in fr_rels]

    en_emb = embed_model.encode([f"query: {t}" for t in en_texts], normalize_embeddings=True)
    fr_emb = embed_model.encode([f"passage: {t}" for t in fr_texts], normalize_embeddings=True)

    sim = en_emb @ fr_emb.T

    used_fr = set()
    shown = 0

    for i in range(len(en_rels)):
        if shown >= max_rel:
            break

        best_j = None
        best_s = -1
        for j in np.argsort(-sim[i]):
            j = int(j)
            if j in used_fr:
                continue
            best_j = j
            best_s = float(sim[i, j])
            break

        if best_j is None:
            continue

        a_en, b_en, lab_en = en_rels[i]
        a_fr, b_fr, lab_fr = fr_rels[best_j]
        used_fr.add(best_j)

        print("-"*120)
        print(f"Match sim={best_s:.3f} | LABEL EN={lab_en} FR={lab_fr}")
        print("EN A:", a_en.strip())
        print("FR A:", a_fr.strip())
        print("EN B:", b_en.strip())
        print("FR B:", b_fr.strip())

        shown += 1

In [ ]:
fr_ids_with_rels = []
for ex in fr_train:
    rels = ast.literal_eval(ex["relations_gen"])
    if len(rels) > 0:
        fr_ids_with_rels.append(ex["id"])

print("Docs with >=1 FR relation:", len(fr_ids_with_rels))

Docs with >=1 FR relation: 379


In [ ]:
for doc_id in random.sample(fr_ids_with_rels, 3):
    show_doc_relations_matched(doc_id, max_rel=5)

DOC: 245_115
EN relations: 6 | FR relations_gen: 6
------------------------------------------------------------------------------------------------------------------------
Match sim=0.903 | LABEL EN=Support FR=Support
EN A: This is a patient undergoing chemotherapy with respiratory symptoms and the "halo sign" on X-ray
FR A: Il s'agit d'un patient sous chimiothérapie présentant des symptômes respiratoires et le "signe du halo" à la radiographie,
EN B: very suggestive of invasive pulmonary aspergillosis
FR B: très évocateur d'une aspergillose pulmonaire invasive.
------------------------------------------------------------------------------------------------------------------------
Match sim=0.914 | LABEL EN=Support FR=Support
EN A: very suggestive of invasive pulmonary aspergillosis
FR A: très évocateur d'une aspergillose pulmonaire invasive.
EN B: Treatment is with voriconazole
FR B: Le traitement consiste à utiliser le voriconazole.
---------------------------------------------------

По выведенным примерам видно, что элементы структуры и метки отношений перенесены адекватно. Теперь можно приступать к обучению модели на подготовленном датасете.

Пример из датасета

In [ ]:
example = ds_en["train"][0]
print(example)

{'id': '402_183', 'text': '[[\'QUESTION\', \'TYPE:\', \'PRIMARY\', \'CARE\', \'AND\', \'SOCIAL\', \'NETWORKS\'], [\'CLINICAL\', \'CASE:\'], [\'Juan,\', \'a\', \'second\', \'year\', \'resident,\', \'attends\', \'Sofia,\', \'a\', \'15\', \'year\', \'old\', \'girl\', \'in\', \'the\', \'emergency\', \'room,\', \'who\', \'apparently\', \'fainted\', \'at\', \'school\', \'without\', \'losing\', \'consciousness.\'], [\'The\', \'patient\', \'says\', \'that\', \'she\', \'was\', \'due\', \'to\', \'take\', \'an\', \'exam,\', \'which\', \'caused\', \'her\', \'a\', \'lot\', \'of\', \'anxiety.\'], [\'From\', \'the\', \'interrogation,\', \'it\', \'appears\', \'that\', \'she\', \'was\', \'being\', \'bullied\', \'by\', \'her\', \'classmates\', \'and\', \'that\', \'she\', \'may\', \'have\', \'an\', \'eating\', \'disorder.\'], [\'Vital\', \'signs\', \'and\', \'neurological\', \'examination\', \'are\', \'normal.\'], [\'Juan\', \'keeps\', \'Soﬁa\', \'under\', \'observation\', \'while\', \'waiting\', \'for\'

*4. Загрузим модель*

In [ ]:
from transformers import pipeline

pipe = pipeline("fill-mask", model="FacebookAI/xlm-roberta-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
label_list = sorted(set(train_flat["label"]))
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}
print("label2id:", label2id)

def add_numeric_label(ex):
    ex["labels"] = label2id[ex["label"]]
    return ex

train_flat2 = train_flat.map(add_numeric_label)
val_flat2   = val_flat.map(add_numeric_label)
test_flat2  = test_flat.map(add_numeric_label)

label2id: {'Attack': 0, 'Support': 1}


Map:   0%|          | 0/9536 [00:00<?, ? examples/s]

Map:   0%|          | 0/1484 [00:00<?, ? examples/s]

Map:   0%|          | 0/3016 [00:00<?, ? examples/s]

In [ ]:
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_LEN = 128

def tokenize_batch(batch):
    return tokenizer(
        batch["text_a"],
        batch["text_b"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

remove_cols = ["text_a", "text_b", "label", "lang", "doc_id"]

train_tok = train_flat2.map(tokenize_batch, batched=True, remove_columns=remove_cols)
val_tok   = val_flat2.map(tokenize_batch, batched=True, remove_columns=remove_cols)
test_tok  = test_flat2.map(tokenize_batch, batched=True, remove_columns=remove_cols)

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/9536 [00:00<?, ? examples/s]

Map:   0%|          | 0/1484 [00:00<?, ? examples/s]

Map:   0%|          | 0/3016 [00:00<?, ? examples/s]

Добавим предобучение на нескольких текстах (возможно, придется добавлять больше текстов))

In [ ]:
unlabeled_texts = [
    "Like many other problems of today, that of cardiovascular diseases is becoming more and more urgent as far as young people are concerne d. Congenital malformations are an obvious paediatric problem. Rheumatic fever and rheumatic heart disease, particularly in hot areas where there is overcrowding and wher e hygienic measures are inadequate, and cardiomyopathies of infectious, parasitic and unknown causes, occur mainly in youth. Hyperte nsion resulting from g lomc rulonephritis or pyelonephritis usually has its roots in childhood. Atherosclerosis in some vessels also begins in childhood, although its most serious complications in the heart and brain usually become manifest in middle age. The present possib ilities for prevention and control at the level of the community are shown in table 1. Enough knowledge already exists for the effective control and prevention of rheumatic heart disease, chronic car pulmonale and heart diseases c: onnectr: d with infections. Essential hypertension can b e effi cie ntly controlled so that its serious complications, such as hypertensive heart disease and cerebrovascular disease, are reduced. Although it is not yet possible to rr e vent the occurrence of ischaemic heart di SP, as e e r peripher a l vascul a r dis e ases of atheroscle rotic origin, t imely tr eatment a nd rehabilitation of subjects with acute myocardial infarction a llows the majority to r e sume previous life activities. These possib\lities are well known and have been summarized also in WHO publications. However, they have not bee n applied adequately in many areas wher e these conditions are common. ",
    "Radical treatment of precancerous conditions is often possible, but in other cases it may be difficult to r e move the lesion completely. Also, the carcinogenic process may continue to act after the precancerous lesion has been removed. This is frequently seen in lesions such as leukoplasias of the mouth and papillomas of the bladder. Patients with definite precancerous conditions should therefore be followed up for a considerable period of time and properly registered, preferably by a national cancer registry. ",
    "Los exámenes de aspirado de médula ósea (MO) muestran un sistema megacariopoyético íntegro o hiperplástico compatible con destrucción plaquetaria periférica, y al igual que en la púrpura trombocitopénica idiopática (PTI), se plantea que los anticuerpos antipla-quetarios, así como el secuestro y destrucción de las plaquetas en el bazo son los responsables de la trombopenia,6 ya que en pacientes afectados con rubéola, influenza y vaccinia se reporta la unión de los virus a las plaquetas de sangre periférica (SP).",
    "Ocurren principalmente en las arterias lenticuloestriadas y talamoperforantes. Aunque se han descrito por lo menos 20 síndromes lacunares, los 5 más frecuentes son: hemiparesia motora pura, síndrome sensitivo puro, síndrome sensitivo-motor, disartria-mano torpe y hemiparesia atáxica. Los principales factores de riesgo asociados a IL son hipertensión arterial (HAS) y diabetes mellitus17-19.Los hallazgos que apoyan la enfermedad de pequeño vaso son: a) síndrome lacunar, b) historia de diabetes o HAS, c) IC menor de 1.5 cm localizado en estructuras profundas y, c) exclusión de otras causas",
    "L’attuale incremento dell’incidenza di diverse patologie a trasmissione sessuale (sifilide, gonorrea, infezione da Chlamydia e, probabilmente, anche infezioni da Herpes e papilloma virus) osservato in alcuni Paesi dell’Unione Europea e comparso dopo un periodo di oltre un decennio in cui l’incidenza era fortemente diminuita, avviene in un contesto di aumento della trasmissione eterosessuale dell’HIV e della prevalenza dell’infezione stessa, in conseguenza della sempre più estesa applicazione della terapia antiretrovirale, che ha aumentato sensibilmente la sopravvivenza dei soggetti affetti da tale patologia.",
    "Fra le malattie infettive di maggiore risalto in ambito occupazionale si ricordano le zoonosi, quali la brucellosi, le rickettsiosi, la leishmaniosi, l’echinococcosi, particolarmente endemiche nelle regioni centro-meridionali italiane, ed altre meno diffuse, ma non trascurabili, quali la borreliosi di Lyme, frequente nelle regioni settentrionali, la febbre Q, la leptospirosi, la tubercolosi da Mycobacterium tuberculosis Avium e Bovis, l’afta epizootica.",
    "Selon une étude récente, les maladies les plus invalidantes en Belgique sont, par ordre décroissant d’importance : les maladies de l’appareil locomoteur, dont les maux de dos et les douleurs cervicales ; les conséquences d’accidents ; la dépression ; le diabète ; la migraine et la bronchite chronique ",
    "Une maladie auto-immune peut se définir par l'activation du système immunitaire du patient contre ses propres antigènes. Les lymphocytes T et B autoréactifs activés vont entraîner la destruction des propres constituants de l'individu via plusieurs mécanismes, soit directement par les lymphocytes T cytotoxiques ou par les dépôts d'anticorps activant le système du complément, soit indirectement via l'activation d'autres cellules impliquées dans la réponse inflammatoire comme au cours des atteintes rénales du lupus. "
]

<>:2: SyntaxWarning: invalid escape sequence '\l'
<>:2: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_615/3037692190.py:2: SyntaxWarning: invalid escape sequence '\l'
  "Like many other problems of today, that of cardiovascular diseases is becoming more and more urgent as far as young people are concerne d. Congenital malformations are an obvious paediatric problem. Rheumatic fever and rheumatic heart disease, particularly in hot areas where there is overcrowding and wher e hygienic measures are inadequate, and cardiomyopathies of infectious, parasitic and unknown causes, occur mainly in youth. Hyperte nsion resulting from g lomc rulonephritis or pyelonephritis usually has its roots in childhood. Atherosclerosis in some vessels also begins in childhood, although its most serious complications in the heart and brain usually become manifest in middle age. The present possib ilities for prevention and control at the level of the community are shown in table 1. Enough knowledg

In [ ]:
from transformers import AutoTokenizer

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": unlabeled_texts})
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

Сохраняем предобученную модель

In [ ]:
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./xlmr-medical-mlm",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    save_steps=10000,
    save_total_limit=2,
    logging_steps=500,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=1.8317138353983562, metrics={'train_runtime': 203.8059, 'train_samples_per_second': 0.118, 'train_steps_per_second': 0.015, 'total_flos': 10244493287424.0, 'train_loss': 1.8317138353983562, 'epoch': 3.0})

In [ ]:
trainer.save_model("./xlmr-medical-adapted")
tokenizer.save_pretrained("./xlmr-medical-adapted")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./xlmr-medical-adapted/tokenizer_config.json',
 './xlmr-medical-adapted/tokenizer.json')

In [ ]:
model_name = "./xlmr-medical-adapted"

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    metrics = {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

    report = classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in range(len(id2label))],
        digits=4,
        zero_division=0,
        output_dict=True
    )

    if "Attack" in report:
        metrics["attack_precision"] = report["Attack"]["precision"]
        metrics["attack_recall"] = report["Attack"]["recall"]
        metrics["attack_f1"] = report["Attack"]["f1-score"]

    if "Support" in report:
        metrics["support_precision"] = report["Support"]["precision"]
        metrics["support_recall"] = report["Support"]["recall"]
        metrics["support_f1"] = report["Support"]["f1-score"]

    return metrics

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: ./xlmr-medical-adapted
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import transformers, inspect
from transformers import TrainingArguments

print("transformers version:", transformers.__version__)

params = inspect.signature(TrainingArguments.__init__).parameters
print("supports evaluation_strategy:", "evaluation_strategy" in params)
print("supports eval_strategy:", "eval_strategy" in params)
print("supports save_strategy:", "save_strategy" in params)
print("supports load_best_model_at_end:", "load_best_model_at_end" in params)
print("supports metric_for_best_model:", "metric_for_best_model" in params)
print("supports warmup_ratio:", "warmup_ratio" in params)


transformers version: 5.3.0
supports evaluation_strategy: False
supports eval_strategy: True
supports save_strategy: True
supports load_best_model_at_end: True
supports metric_for_best_model: True
supports warmup_ratio: True


In [ ]:
import transformers, inspect
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./xlmr_multilingual_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
     save_total_limit=1,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Attack Precision,Attack Recall,Attack F1,Support Precision,Support Recall,Support F1
1,0.215163,0.433128,0.814016,0.771194,0.628889,0.721939,0.672209,0.894584,0.847070,0.870179
2,0.197307,0.759701,0.811321,0.774156,0.614286,0.767857,0.682540,0.908451,0.826923,0.865772
3,0.099017,0.776000,0.831536,0.789999,0.664352,0.732143,0.696602,0.900190,0.867216,0.883396


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1788, training_loss=0.20700929842272595, metrics={'train_runtime': 1305.2377, 'train_samples_per_second': 21.918, 'train_steps_per_second': 1.37, 'total_flos': 1881770267934720.0, 'train_loss': 0.20700929842272595, 'epoch': 3.0})

Проведём проверку на тестовых данных.

In [ ]:
test_output = trainer.predict(test_tok)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

pred_output = trainer.predict(test_tok)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

test_metrics = compute_metrics((pred_output.predictions, pred_output.label_ids))
print("Test metrics:\n")
print(test_metrics)

print("\nClassification report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))],
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{id2label[i]}" for i in range(len(id2label))],
    columns=[f"Pred_{id2label[i]}" for i in range(len(id2label))]
)

print("Confusion matrix:\n")
display(cm_df)

Test metrics:

{'accuracy': 0.8395225464190982, 'f1_macro': 0.8113221399304613, 'attack_precision': 0.7674157303370787, 'attack_recall': 0.7114583333333333, 'attack_f1': 0.7383783783783784, 'support_precision': 0.8697083725305739, 'support_recall': 0.89931906614786, 'support_f1': 0.8842659014825442}

Classification report:

              precision    recall  f1-score   support

      Attack     0.7674    0.7115    0.7384       960
     Support     0.8697    0.8993    0.8843      2056

    accuracy                         0.8395      3016
   macro avg     0.8186    0.8054    0.8113      3016
weighted avg     0.8371    0.8395    0.8378      3016

Confusion matrix:



,Pred_Attack,Pred_Support
True_Attack,683,277
True_Support,207,1849


Результаты работы baseline модели оказались неплохими. Теперь можно приступить к fine tune.

Добавим взвешенную функцию потерь, чтобы справиться с дисбалансом классов.

Посчитаем веса классов.

In [ ]:
import torch.nn as nn
from transformers import Trainer
from collections import Counter

y_train = np.array(train_tok["labels"])
class_counts = Counter(y_train)

print("Class counts:", class_counts)
print("id2label:", id2label)

num_classes = len(id2label)
total_samples = len(y_train)

class_weights = torch.tensor(
    [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float
)

print("Class weights:", class_weights)

Class counts: Counter({np.int64(1): 6504, np.int64(0): 3032})
id2label: {0: 'Attack', 1: 'Support'}
Class weights: tensor([1.5726, 0.7331])


Задаём новую функцию обучения.

In [ ]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

Создадим копию модели для обучения.

In [ ]:
from transformers import AutoModelForSequenceClassification

model_weighted = AutoModelForSequenceClassification.from_pretrained(
    "./xlmr-medical-adapted",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: ./xlmr-medical-adapted
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args_weighted = TrainingArguments(
    output_dir="./xlmr_weighted",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=150,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
)

trainer_weighted = WeightedTrainer(
    class_weights=class_weights,
    model=model_weighted,
    args=training_args_weighted,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

trainer_weighted.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Attack Precision,Attack Recall,Attack F1,Support Precision,Support Recall,Support F1
1,0.568336,0.462884,0.815364,0.772196,0.632287,0.719388,0.673031,0.894027,0.849817,0.871362
2,0.314329,0.512254,0.819407,0.787112,0.620623,0.813776,0.704194,0.924742,0.821429,0.870029
3,0.193478,0.737395,0.845687,0.807782,0.688222,0.760204,0.722424,0.910561,0.876374,0.893140


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1788, training_loss=0.3870411132539412, metrics={'train_runtime': 1431.0514, 'train_samples_per_second': 19.991, 'train_steps_per_second': 1.249, 'total_flos': 1881770267934720.0, 'train_loss': 0.3870411132539412, 'epoch': 3.0})

Посмотрим, улучшилось ли качество обработки после добавления взвешенной функции потерь.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

test_metrics_weighted = trainer_weighted.predict(test_tok)
print("Weighted test metrics:", test_metrics_weighted)

pred_output = trainer_weighted.predict(test_tok)
y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print("\nClassification report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
))

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f"True_{id2label[i]}" for i in range(len(id2label))],
    columns=[f"Pred_{id2label[i]}" for i in range(len(id2label))]
)

print("\nConfusion matrix:\n")
print(cm_df)

Weighted test metrics: PredictionOutput(predictions=array([[-2.87475  ,  2.6452115],
       [-2.7471733,  2.524373 ],
       [-2.9761286,  2.7567232],
       ...,
       [-3.1501017,  2.9653792],
       [-2.878904 ,  2.6717265],
       [ 1.3257103, -1.6240437]], dtype=float32), label_ids=array([1, 1, 1, ..., 1, 1, 1]), metrics={'test_loss': 0.7795882821083069, 'test_accuracy': 0.8448275862068966, 'test_f1_macro': 0.81875, 'test_attack_precision': 0.7697368421052632, 'test_attack_recall': 0.73125, 'test_attack_f1': 0.75, 'test_support_precision': 0.8773764258555133, 'test_support_recall': 0.8978599221789884, 'test_support_f1': 0.8875, 'test_runtime': 22.0402, 'test_samples_per_second': 136.841, 'test_steps_per_second': 4.31})



Classification report:

              precision    recall  f1-score   support

      Attack       0.77      0.73      0.75       960
     Support       0.88      0.90      0.89      2056

    accuracy                           0.84      3016
   macro avg       0.82      0.81      0.82      3016
weighted avg       0.84      0.84      0.84      3016


Confusion matrix:

              Pred_Attack  Pred_Support
True_Attack           702           258
True_Support          210          1846
